In [4]:
from pathlib import Path

# CELL 1: Configuration - recursive Oracle documentation roots by platform
# Each TOC URL is crawled recursively within its own Oracle documentation guide directory.
PLATFORM_URLS = {
    "Exadata On-Prem": [
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmin/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/sagug/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmso/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmsq/toc.htm",
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmmn/toc.htm",
    ],
    "Exascale": [
        "https://docs.oracle.com/en/engineered-systems/exadata-database-machine/exscl/toc.htm",
    ],
    "ExaCC": [
        "https://docs.oracle.com/en/engineered-systems/exadata-cloud-at-customer/ecccm/toc.htm",
    ],
    "ExaCS": [
        "https://docs.oracle.com/en/engineered-systems/exadata-cloud-service/ecscm/toc.htm",
    ],
}

MAX_DEPTH = 8
MAX_PAGES = 3000
REQUEST_TIMEOUT = 30
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Oracle Knowledge article ingestion (KM.xlsx / KM_updated.xlsx)
KM_EXCEL_FILE = "KM_updated.xlsx" if Path("KM_updated.xlsx").exists() else "KM.xlsx"  # workbook containing the ID column
KM_ID_COLUMN = "ID"
KM_URL_TEMPLATE = (
    "https://fa-etmi-saasfaprod1.fa.ocs.oraclecloud.com/"
    "fscmUI/redwood/service/ec/container/sr/view-article?docId={}"
)
KM_AUTH_PROFILE = "./oracle_km_browser_profile"
KM_WAIT_AFTER_LOAD = 5
KM_MAX_ARTICLES = None  # e.g. 10 for a test run; None = all IDs


In [5]:
%pip install openpyxl playwright beautifulsoup4 lxml
!playwright install chromium

  Using cached playwright-1.63.0-py3-none-macosx_11_0_arm64.whl.metadata (3.3 kB)
  Using cached lxml-6.1.3-cp313-cp313-macosx_10_13_universal2.whl.metadata (3.3 kB)
  Using cached pyee-13.0.1-py3-none-any.whl.metadata (3.0 kB)
Using cached playwright-1.63.0-py3-none-macosx_11_0_arm64.whl (42.9 MB)
Using cached pyee-13.0.1-py3-none-any.whl (15 kB)
Using cached lxml-6.1.3-cp313-cp313-macosx_10_13_universal2.whl (8.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openpyxl]5/6 [openpyxl]t]
Note: you may need to restart the kernel to use updated packages.


In [6]:
# CELL 2: Imports and recursive URL crawler
import os
import re
import shutil
from io import BytesIO
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag, parse_qsl, urlencode

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from langchain_core.documents import Document

def normalize_url(url):
    url, _ = urldefrag(url)
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return None
    # Drop tracking/query parameters while retaining meaningful query URLs.
    kept_query = [(k, v) for k, v in parse_qsl(parsed.query)
                  if not k.lower().startswith(("utm_", "source", "ref"))]
    return parsed._replace(query=urlencode(kept_query), fragment="").geturl()

def scope_prefix(root_url):
    parsed = urlparse(root_url)
    path = parsed.path or "/"
    if not path.endswith("/"):
        path = path.rsplit("/", 1)[0] + "/"
    return (parsed.scheme, parsed.netloc, path)

def in_scope(url, root_scope):
    parsed = urlparse(url)
    scheme, netloc, prefix = root_scope
    return parsed.scheme == scheme and parsed.netloc == netloc and parsed.path.startswith(prefix)

def extract_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    links = [urljoin]  # placeholder removed below; keeps parser logic easy to read
    return text, soup

def crawl_platform(root_url, platform_name, max_depth=MAX_DEPTH, max_pages=MAX_PAGES):
    root_url = normalize_url(root_url)
    if not root_url:
        raise ValueError(f"Invalid root URL: {root_url}")

    root_scope = scope_prefix(root_url)
    queue = deque([(root_url, 0)])
    queued = {root_url}
    visited = set()
    documents = []
    session = requests.Session()
    session.headers.update({"User-Agent": "Oracle-Exadata-RAG-Doc-Crawler/1.0"})

    while queue and len(visited) < max_pages:
        url, depth = queue.popleft()
        if url in visited:
            continue
        visited.add(url)

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            response.raise_for_status()
        except requests.RequestException as exc:
            print(f"WARN: {url} -> {exc}")
            continue

        final_url = normalize_url(response.url) or url
        content_type = response.headers.get("content-type", "").lower()
        is_pdf = "application/pdf" in content_type or final_url.lower().endswith(".pdf")

        if is_pdf:
            try:
                reader = PdfReader(BytesIO(response.content))
                for page_num, page in enumerate(reader.pages):
                    text = (page.extract_text() or "").strip()
                    if text:
                        documents.append(Document(
                            page_content=text,
                            metadata={
                                "source": final_url,
                                "platform": platform_name,
                                "page": page_num,
                                "content_type": "pdf",
                            },
                        ))
            except Exception as exc:
                print(f"WARN: PDF parse failed for {final_url}: {exc}")
            continue

        if "text/html" not in content_type:
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()
        text = soup.get_text("\n", strip=True)
        text = re.sub(r"\n{3,}", "\n\n", text)
        if text:
            documents.append(Document(
                page_content=text,
                metadata={
                    "source": final_url,
                    "platform": platform_name,
                    "depth": depth,
                    "content_type": "html",
                },
            ))

        if depth >= max_depth:
            continue

        for anchor in soup.find_all("a", href=True):
            child = normalize_url(urljoin(final_url, anchor["href"]))
            if child and child not in visited and child not in queued and in_scope(child, root_scope):
                queued.add(child)
                queue.append((child, depth + 1))

    print(f"{platform_name}: visited={len(visited)}, queue_remaining={len(queue)}, documents={len(documents)}")
    return documents

# --- Oracle Knowledge article helpers ---
from pathlib import Path
from openpyxl import load_workbook
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError


def read_km_ids(excel_file=KM_EXCEL_FILE, id_column=KM_ID_COLUMN):
    """Read unique Knowledge article IDs from the workbook's ID column."""
    wb = load_workbook(excel_file, read_only=True, data_only=True)
    ids = []
    seen = set()

    try:
        for ws in wb.worksheets:
            headers = {
                str(cell.value).strip(): cell.column
                for cell in ws[1]
                if cell.value is not None
            }
            if id_column not in headers:
                continue

            id_col = headers[id_column]
            for row in range(2, ws.max_row + 1):
                value = ws.cell(row=row, column=id_col).value
                if value is None:
                    continue
                doc_id = str(value).strip()
                if doc_id and doc_id not in seen:
                    seen.add(doc_id)
                    ids.append(doc_id)
    finally:
        wb.close()

    return ids


def _clean_article_text(root):
    """Convert an article DOM subtree into compact, indexable text."""
    soup = BeautifulSoup(str(root), "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "nav", "footer"]):
        tag.decompose()

    text = soup.get_text("\n", strip=True)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()



async def _extract_article(page, doc_id, url):
    """Extract title + article content from the rendered Oracle page."""
    title = ""
    try:
        title = (await page.locator("h1").first.inner_text(timeout=5000)).strip()
    except Exception:
        pass

    candidates = [
        "article",
        "main",
        '[role="main"]',
        '[data-testid*="article" i]',
        '[class*="article" i]',
        '[class*="knowledge" i]',
    ]

    best_text = ""
    for selector in candidates:
        try:
            loc = page.locator(selector)
            count = await loc.count()
            for i in range(min(count, 5)):
                text = _clean_article_text(await loc.nth(i).inner_html())
                if len(text) > len(best_text):
                    best_text = text
        except Exception:
            continue

    if not best_text:
        best_text = _clean_article_text(await page.locator("body").inner_html())

    if title and title not in best_text:
        best_text = f"{title}\n\n{best_text}"

    return Document(
        page_content=best_text,
        metadata={
            "source": url,
            "platform": "Oracle Knowledge",
            "doc_id": doc_id,
            "title": title,
            "content_type": "oracle_knowledge_article",
        },
    )


async def load_oracle_km_documents(
    excel_file=KM_EXCEL_FILE,
    auth_profile=KM_AUTH_PROFILE,
    max_articles=KM_MAX_ARTICLES,
):
    """
    Open Oracle Knowledge articles one-by-one in Chromium and return LangChain Documents.

    First run: a visible browser opens so the user can complete Oracle SSO.
    The persistent browser profile keeps that session for later notebook runs.
    """
    doc_ids = read_km_ids(excel_file)
    if max_articles is not None:
        doc_ids = doc_ids[:max_articles]

    print(f"Oracle Knowledge IDs found: {len(doc_ids)}")
    if not doc_ids:
        return []

    documents = []

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=auth_profile,
            headless=False,
            accept_downloads=False,
            viewport={"width": 1440, "height": 1000},
        )
        page = await context.new_page()
        page.set_default_timeout(60000)

        first_url = KM_URL_TEMPLATE.format(doc_ids[0])
        await page.goto(first_url, wait_until="domcontentloaded", timeout=60000)

        print("\nOracle browser opened.")
        print("Complete Oracle SSO/login in the browser if prompted.")
        input("After the article is visible and you are authenticated, press ENTER here to start extraction... ")

        for i, doc_id in enumerate(doc_ids, 1):
            url = KM_URL_TEMPLATE.format(doc_id)
            print(f"[{i}/{len(doc_ids)}] {doc_id}")

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                await page.wait_for_timeout(KM_WAIT_AFTER_LOAD * 1000)

                try:
                    await page.wait_for_load_state("networkidle", timeout=15000)
                except PlaywrightTimeoutError:
                    pass

                current_url = page.url.lower()
                if "login" in current_url or "signin" in current_url:
                    print(f"  AUTH REQUIRED - stopping at {doc_id}")
                    break

                doc = await _extract_article(page, doc_id, url)
                if doc.page_content.strip():
                    documents.append(doc)
                    print(f"  extracted {len(doc.page_content):,} characters")
                else:
                    print("  WARN: empty article content")

            except Exception as exc:
                print(f"  WARN: failed {doc_id}: {exc}")

        await context.close()

    return documents


In [7]:
# CELL 3: Crawl every configured root recursively
all_docs = []
for platform_name, root_urls in PLATFORM_URLS.items():
    print(f"\n=== Crawling {platform_name} ===")
    for root_url in root_urls:
        print(f"Root: {root_url}")
        platform_docs = crawl_platform(root_url, platform_name)
        all_docs.extend(platform_docs)

print(f"\nTotal source documents collected: {len(all_docs)}")



=== Crawling Exadata On-Prem ===
Root: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmin/toc.htm
Exadata On-Prem: visited=328, queue_remaining=0, documents=328
Root: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/sagug/toc.htm
Exadata On-Prem: visited=548, queue_remaining=0, documents=548
Root: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmso/toc.htm
Exadata On-Prem: visited=556, queue_remaining=0, documents=556
Root: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmsq/toc.htm
Exadata On-Prem: visited=87, queue_remaining=0, documents=87
Root: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmmn/toc.htm
WARN: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmmn/cell-server-charts.html -> 404 Client Error: Not Found for url: https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmmn/cell-server-charts.html
Exadata On-Pr

In [8]:
# CELL 4: Load Oracle Knowledge articles from the ID column and merge with crawler documents
# Set KM_MAX_ARTICLES to a small number in the configuration cell for a test run.
km_docs = await load_oracle_km_documents()
all_docs.extend(km_docs)

print(f"Oracle Knowledge documents added: {len(km_docs)}")
print(f"Total source documents before chunking: {len(all_docs)}")

if km_docs:
    print("Sample KM metadata:", km_docs[0].metadata)
    print("Sample KM text:\n", km_docs[0].page_content[:1000])


Oracle Knowledge IDs found: 97

Oracle browser opened.
Complete Oracle SSO/login in the browser if prompted.
[1/97] FAQ7941
  extracted 28,571 characters
[2/97] KB930763
  extracted 3,460 characters
[3/97] KB153930
  extracted 82,575 characters
[4/97] KI55547
  extracted 2,502 characters
[5/97] KB290498
  extracted 1,096 characters
[6/97] KB930449
  extracted 33 characters
[7/97] KB114881
  extracted 10,241 characters
[8/97] KB492866
  extracted 16,854 characters
[9/97] KB467937
  extracted 26,489 characters
[10/97] KB914580
  extracted 3,707 characters
[11/97] PALRT2778
  extracted 12,453 characters
[12/97] KB403624
  extracted 21,489 characters
[13/97] KB623062
  extracted 35,413 characters
[14/97] KB922069
  extracted 4,485 characters
[15/97] KB858617
  extracted 5,106 characters
[16/97] KB929331
  extracted 1,184 characters
[17/97] KB922070
  extracted 4,409 characters
[18/97] KB928867
  extracted 33 characters
[19/97] KB928787
  extracted 1,346 characters
[20/97] KB928654
  extrac

In [13]:
# CELL 5: Validate Oracle Knowledge ingestion metadata before chunking
km_chunk_count = sum(
    1 for d in docs_chunks if d.metadata.get("content_type") == "oracle_knowledge_article"
)
print(f"Oracle Knowledge chunks: {km_chunk_count}")

if km_docs:
    assert all(d.metadata.get("doc_id") for d in km_docs), "Missing doc_id metadata"
    assert all(d.metadata.get("source", "").startswith("https://") for d in km_docs), "Missing source URL"
    print("KM metadata validation passed.")


Oracle Knowledge chunks: 967
KM metadata validation passed.


In [14]:
# CELL 6: Split all documents while preserving URL and article metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

docs_chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks generated: {len(docs_chunks)}")
print("Sample metadata:", docs_chunks[0].metadata if docs_chunks else "No documents")


Total chunks generated: 15457
Sample metadata: {'source': 'https://docs.oracle.com/en/engineered-systems/exadata-database-machine/dbmin/toc.htm', 'platform': 'Exadata On-Prem', 'depth': 0, 'content_type': 'html'}


In [15]:
# CELL 7: Local HuggingFace embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17661.16it/s]


Embedding model ready.


In [17]:
# CELL 8: Build Oracle Knowledge Chroma database

import os
import shutil
from pathlib import Path
from langchain_chroma import Chroma

# Use a fresh directory to avoid stale/readonly Chroma files
DB_FOLDER = Path("./vectordb_oracle_knowledge")

# Remove an old database if it exists
if DB_FOLDER.exists():
    print(f"Removing old Chroma database: {DB_FOLDER}")
    shutil.rmtree(DB_FOLDER)

DB_FOLDER.mkdir(parents=True, exist_ok=True)

# Only index Oracle Knowledge documents
oracle_chunks = [
    d for d in docs_chunks
    if d.metadata.get("platform") == "Oracle Knowledge"
]

print(f"Oracle Knowledge chunks: {len(oracle_chunks)}")

if not oracle_chunks:
    raise ValueError("No Oracle Knowledge chunks found.")

# Create Chroma
oracle_db = Chroma.from_documents(
    documents=oracle_chunks,
    embedding=embedding,
    persist_directory=str(DB_FOLDER),
    collection_name="oracle_knowledge",
)

print(f"Chroma database created successfully: {DB_FOLDER}")
print(f"Indexed chunks: {len(oracle_chunks)}")

Oracle Knowledge chunks: 967
Chroma database created successfully: vectordb_oracle_knowledge
Indexed chunks: 967


In [18]:
# CELL 10: Optional quick sanity check
for platform_name, folder in DB_FOLDERS.items():
    if os.path.exists(folder):
        test_db = Chroma(persist_directory=folder, embedding_function=embedding, collection_name="exadata_docs")
        print(platform_name, "->", test_db._collection.count(), "chunks")


Oracle Knowledge -> 967 chunks
Exadata On-Prem -> 0 chunks
Exascale -> 1155 chunks
ExaCC -> 3001 chunks
ExaCS -> 2902 chunks


In [19]:
import sys
import chromadb

print("Python:", sys.executable)
print("Chroma version:", chromadb.__version__)

client = chromadb.PersistentClient(
    path="./vectordb_oracle_knowledge"
)

collections = client.list_collections()

print("Number of collections:", len(collections))

for c in collections:
    print("NAME :", c.name)
    print("COUNT:", c.count())

Python: /Users/aneesmaniyar/Downloads/ML_OPS/exadataragchatbot/.venv/bin/python
Chroma version: 1.5.9
Number of collections: 1
NAME : oracle_knowledge
COUNT: 967
